# 🏥 Aged Care Demand Forecasting — Australian Public Sector
## Notebook 04: Feature Engineering & Preprocessing

> **Inputs from 02_data_collection_REAL.ipynb:**
> - `seifa_2021.csv` — IRSD, IRSAD scores + remoteness (SA2)
> - `abs_population_projections.csv` — pop_70plus, projections 2031/2041, state (SA2)
> - `aihw_recipients_sa2.csv` — CHSP, home care, residential recipients (SA2)
>
> **No supply file, no quarterly recipients** — supply features derived from
> recipient counts + population as designed in notebook 01.

---

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from pathlib import Path
import joblib
import warnings
warnings.filterwarnings('ignore')

RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# ── Load real data ─────────────────────────────────────────────────────────────
pop_df   = pd.read_csv(RAW_DIR / 'abs_population_projections.csv', dtype={'sa2_code': str})
recv_df  = pd.read_csv(RAW_DIR / 'aihw_recipients_sa2.csv',        dtype={'sa2_code': str})
seifa_df = pd.read_csv(RAW_DIR / 'seifa_2021.csv',                 dtype={'sa2_code': str})

for df in [pop_df, recv_df, seifa_df]:
    df['sa2_code'] = df['sa2_code'].str.zfill(9)

print('✅ Raw data loaded')
print(f'   pop_df:    {pop_df.shape}   columns: {list(pop_df.columns)}')
print(f'   recv_df:   {recv_df.shape}   columns: {list(recv_df.columns)}')
print(f'   seifa_df:  {seifa_df.shape}   columns: {list(seifa_df.columns)}')

---
## 1. Build Master Dataset

In [ ]:
# Base: population (2,454 SA2s)
# Left join recipients — SA2s with no AIHW match get 0
# Left join SEIFA — ~100 unpopulated/suppressed SA2s may be missing
master = (
    pop_df
    .merge(
        recv_df[['sa2_code', 'chsp_recipients', 'home_care_recipients',
                 'residential_recipients', 'total_recipients']],
        on='sa2_code', how='left'
    )
    .merge(
        seifa_df[['sa2_code', 'irsd_score', 'irsd_decile',
                  'irsad_score', 'irsad_decile', 'remoteness_cat']],
        on='sa2_code', how='left'
    )
)

# Fill missing recipient counts with 0
for col in ['chsp_recipients', 'home_care_recipients',
            'residential_recipients', 'total_recipients']:
    master[col] = master[col].fillna(0).astype(int)

print(f'✅ Master dataset: {master.shape}')
print(f'   SA2s with SEIFA:      {master["irsd_score"].notna().sum():,}')
print(f'   SA2s with recipients: {(master["total_recipients"] > 0).sum():,}')
master.head(3)

---
## 2. Demand-Side Features

Based on `pop_70plus` — the demand proxy defined in notebook 01
(70+ is the primary aged care utilisation cohort).

In [ ]:
# Share of population aged 70+
master['pct_70plus_2024'] = (
    master['pop_70plus'] / master['total_pop'].replace(0, np.nan)
).fillna(0).round(4)

# Share aged 65+
master['pct_65plus_2024'] = (
    master['pop_65plus'] / master['total_pop'].replace(0, np.nan)
).fillna(0).round(4)

# Absolute projected growth in 70+ population by 2031
master['pop_70plus_growth_abs'] = (
    master['pop_70plus_2031'] - master['pop_70plus']
).clip(lower=0)

# Growth rate 2024→2031 (already in pop_df as growth_rate_70plus_2031)
# Verify column exists
assert 'growth_rate_70plus_2031' in master.columns,     "growth_rate_70plus_2031 missing — check abs_population_projections.csv"

# Dependency-style ratio: 70+ relative to working-age proxy (total - 70+)
master['elderly_dependency_ratio'] = (
    master['pop_70plus'] / (master['total_pop'] - master['pop_70plus']).replace(0, np.nan)
).fillna(0).round(4)

print('✅ Demand-side features created')
master[['sa2_name', 'pop_70plus', 'pct_70plus_2024', 'pop_70plus_2031',
        'growth_rate_70plus_2031', 'pop_70plus_growth_abs']].head(5)

---
## 3. Supply-Side Features (Derived from Recipients + Population)

No raw supply file is available at SA2 level (AIHW publishes supply at ACPR level only).
Recipients per 1,000 elderly is used as a utilisation/supply proxy per notebook 01 design.

In [ ]:
# Utilisation rates per 1,000 pop_70plus
master['residential_per_1000'] = (
    master['residential_recipients'] /
    master['pop_70plus'].replace(0, np.nan) * 1000
).fillna(0).round(2)

master['home_care_per_1000'] = (
    master['home_care_recipients'] /
    master['pop_70plus'].replace(0, np.nan) * 1000
).fillna(0).round(2)

master['chsp_per_1000'] = (
    master['chsp_recipients'] /
    master['pop_70plus'].replace(0, np.nan) * 1000
).fillna(0).round(2)

master['total_per_1000'] = (
    master['total_recipients'] /
    master['pop_70plus'].replace(0, np.nan) * 1000
).fillna(0).round(2)

# National median utilisation rate — used as benchmark
NAT_MEDIAN_UTIL = master[master['total_per_1000'] > 0]['total_per_1000'].median()
print(f'National median utilisation rate: {NAT_MEDIAN_UTIL:.1f} per 1,000 pop 70+')

# Service gap: how far below national median is each SA2?
# Positive = under-utilised relative to national median (potential unmet need)
master['utilisation_gap'] = (
    NAT_MEDIAN_UTIL - master['total_per_1000']
).round(2)

# Composite service gap score (clipped at 0 — only count under-supply)
master['service_gap_score'] = master['utilisation_gap'].clip(lower=0).round(2)

print('✅ Supply-side features created')
master[['sa2_name', 'residential_per_1000', 'home_care_per_1000',
        'chsp_per_1000', 'total_per_1000', 'service_gap_score']].head(5)

---
## 4. Projected Demand Gap (2031 Forward-Looking)

In [ ]:
# Project recipient counts to 2031 assuming current utilisation rate holds
# per SA2 (uses each SA2's own rate, not national median)
master['utilisation_rate_2024'] = (
    master['total_recipients'] /
    master['pop_70plus'].replace(0, np.nan)
).fillna(NAT_MEDIAN_UTIL / 1000)

master['projected_recipients_2031'] = (
    master['pop_70plus_2031'] * master['utilisation_rate_2024']
).round(0).astype(int)

master['projected_demand_growth'] = (
    master['projected_recipients_2031'] - master['total_recipients']
).clip(lower=0)

# Projected demand growth rate
master['projected_demand_growth_rate'] = (
    master['projected_recipients_2031'] /
    master['total_recipients'].replace(0, np.nan) - 1
).fillna(master['growth_rate_70plus_2031']).round(4)

print(f'Total current recipients:          {master["total_recipients"].sum():,}')
print(f'Total projected recipients 2031:   {master["projected_recipients_2031"].sum():,}')
print(f'Total projected demand growth:     {master["projected_demand_growth"].sum():,}')
print('✅ Projected demand features created')

---
## 5. Target Variable — Demand Risk Label

Rule-based classification using three factors:
- **Growth pressure**: projected 70+ growth rate 2024→2031
- **Service gap**: current utilisation gap vs national median
- **Disadvantage**: IRSD score (lower = more disadvantaged)

In [ ]:
# Compute percentile thresholds for rule calibration
growth_75  = master['growth_rate_70plus_2031'].quantile(0.75)
gap_75     = master['service_gap_score'].quantile(0.75)
gap_50     = master['service_gap_score'].quantile(0.50)
irsd_25    = master['irsd_score'].quantile(0.25)   # bottom quartile = most disadvantaged

print(f'Growth rate 75th pct:  {growth_75:.3f}')
print(f'Service gap 75th pct:  {gap_75:.1f}')
print(f'Service gap 50th pct:  {gap_50:.1f}')
print(f'IRSD score 25th pct:   {irsd_25:.1f}')

def assign_risk_label(row):
    """
    HIGH   — high growth pressure AND significant service gap AND disadvantaged
    MEDIUM — moderate on any two dimensions
    LOW    — well-served, slow growth, not highly disadvantaged
    """
    high_growth = row['growth_rate_70plus_2031'] >= growth_75
    high_gap    = row['service_gap_score'] >= gap_75
    mid_gap     = row['service_gap_score'] >= gap_50
    disadvantaged = (row['irsd_score'] <= irsd_25) if pd.notna(row['irsd_score']) else False
    remote      = row['remoteness_cat'] >= 3 if pd.notna(row['remoteness_cat']) else False

    if high_growth and high_gap and (disadvantaged or remote):
        return 'High'
    elif (high_growth and mid_gap) or (high_gap and disadvantaged) or (remote and high_growth):
        return 'Medium'
    else:
        return 'Low'

master['demand_risk_label'] = master.apply(assign_risk_label, axis=1)

risk_counts = master['demand_risk_label'].value_counts()
print('\nDemand Risk Label Distribution:')
for label in ['High', 'Medium', 'Low']:
    count = risk_counts.get(label, 0)
    pct = count / len(master) * 100
    print(f'  {label:<8} {count:>5}  ({pct:.1f}%)')

fig, ax = plt.subplots(figsize=(6, 3))
colors = {'High': '#C44E52', 'Medium': '#DD8452', 'Low': '#55A868'}
risk_counts.reindex(['High', 'Medium', 'Low']).plot(
    kind='bar', ax=ax,
    color=[colors[l] for l in ['High', 'Medium', 'Low']],
    edgecolor='white', rot=0
)
ax.set_title('Target Variable: Demand Risk Label', fontweight='bold')
ax.set_ylabel('Number of SA2 regions')
for i, label in enumerate(['High', 'Medium', 'Low']):
    count = risk_counts.get(label, 0)
    ax.text(i, count + 1, str(count), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(Path('../reports') / 'fe_01_risk_labels.png', dpi=150)
plt.show()

---
## 6. Final Feature Set

In [ ]:
FEATURE_COLS = [
    # ── Demographic ───────────────────────────────────────────────
    'pop_70plus',                   # base elderly population
    'pct_70plus_2024',              # elderly share of SA2 population
    'pct_65plus_2024',              # broader elderly share
    'elderly_dependency_ratio',     # 70+ relative to rest of population
    'growth_rate_70plus_2031',      # ABS Series B projected growth rate
    'pop_70plus_growth_abs',        # absolute growth in 70+ by 2031
    # ── Recipients / Utilisation ──────────────────────────────────
    'total_recipients',             # current total recipients
    'residential_recipients',       # residential care count
    'home_care_recipients',         # home care package count
    'chsp_recipients',              # CHSP count
    'total_per_1000',               # total utilisation rate
    'residential_per_1000',         # residential utilisation rate
    'home_care_per_1000',           # HCP utilisation rate
    'chsp_per_1000',                # CHSP utilisation rate
    # ── Service Gap ───────────────────────────────────────────────
    'service_gap_score',            # gap vs national median utilisation
    'projected_recipients_2031',    # projected demand 2031
    'projected_demand_growth',      # absolute demand increase by 2031
    'projected_demand_growth_rate', # relative demand growth
    # ── Socio-economic ────────────────────────────────────────────
    'irsd_score',                   # disadvantage index
    'irsd_decile',                  # disadvantage decile
    'irsad_score',                  # advantage/disadvantage index
    'remoteness_cat',               # 1=Major City … 5=Very Remote
]

TARGET_COL = 'demand_risk_label'

print(f'✅ Feature set: {len(FEATURE_COLS)} features')
print(f'   Target: {TARGET_COL}')

# Missing value report
missing = master[FEATURE_COLS].isnull().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    print('   No missing values in feature set ✅')
else:
    print(f'\n⚠️ Missing values:')
    print(missing)
    print('\nImputing missing SEIFA values with national median...')
    for col in ['irsd_score', 'irsd_decile', 'irsad_score', 'remoteness_cat']:
        if col in master.columns:
            master[col] = master[col].fillna(master[col].median())
    print('   Imputation complete ✅')

---
## 7. Encoding & Train/Test Split

In [ ]:
label_map = {'Low': 0, 'Medium': 1, 'High': 2}
master['risk_encoded'] = master[TARGET_COL].map(label_map)

X = master[FEATURE_COLS].copy()
y = master['risk_encoded'].copy()

# Drop rows where target is NaN (shouldn't happen but safety check)
valid = y.notna()
X, y = X[valid], y[valid]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale (for algorithms that need it — XGBoost doesn't but saved for reference)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train set: {X_train.shape[0]:,} samples')
print(f'Test set:  {X_test.shape[0]:,} samples')
print(f'\nClass distribution in train set:')
for label, enc in label_map.items():
    count = (y_train == enc).sum()
    pct = count / len(y_train) * 100
    print(f'  {label:<8} {count:>5}  ({pct:.1f}%)')

---
## 8. Missing Value & Distribution Check

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

plot_cols = [
    'pct_70plus_2024', 'growth_rate_70plus_2031', 'total_per_1000',
    'service_gap_score', 'irsd_score', 'remoteness_cat'
]
for i, col in enumerate(plot_cols):
    axes[i].hist(master[col].dropna(), bins=30, color='#4C72B0',
                 edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('SA2 count')

plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(Path('../reports') / 'fe_02_distributions.png', dpi=150)
plt.show()

---
## 9. Save Processed Data

In [ ]:
# Save master feature dataset
master_out = master[
    ['sa2_code', 'sa2_name', 'state'] + FEATURE_COLS +
    [TARGET_COL, 'risk_encoded']
].copy()
master_out.to_csv(PROCESSED_DIR / 'master_features.csv', index=False)

# Save train/test splits
X_train.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test.to_csv( PROCESSED_DIR / 'X_test.csv',  index=False)
y_train.to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_test.to_csv( PROCESSED_DIR / 'y_test.csv',  index=False)

# Save scaler and metadata
joblib.dump(scaler, PROCESSED_DIR / 'scaler.pkl')
pd.Series(FEATURE_COLS).to_csv(PROCESSED_DIR / 'feature_cols.csv', index=False, header=False)
pd.Series(label_map).to_csv(PROCESSED_DIR / 'label_map.csv', header=False)

print('=== Processed Data Saved ===')
for f in sorted(PROCESSED_DIR.glob('*')):
    kb = f.stat().st_size / 1024
    print(f'  {f.name:<35} ({kb:.1f} KB)')

print(f'\n✅ Feature engineering complete')
print(f'   Master features: {master_out.shape}')
print(f'   Train/test: {X_train.shape[0]} / {X_test.shape[0]} samples')
print(f'\n➡️  Next: 05_modelling.ipynb')